# 🔁 Рекуррентные сети: нейросеть с памятью

> Сначала: **Файл → Сохранить копию на Диске**. Затем: **Среда выполнения → Сменить среду → GPU**.

Вы читаете текст **по одному слову** и держите в голове, о чём он. Каждое новое слово чуть меняет вашу «заметку в голове». Рекуррентная сеть делает то же самое:

```
слово 1 → [ячейка] → память 1
слово 2 → [ячейка] → память 2   (ячейка видит слово 2 и память 1)
слово 3 → [ячейка] → память 3   (видит слово 3 и память 2)
                       ↓
              по последней памяти решаем: спорт, наука или экономика
```

План: **память своими руками → слой `nn.RNN` → классификатор на RNN → LSTM → сравнение**.

Каждое задание: замените `None` на свой код → **Shift + Enter** → запустите ячейку 🧪 под ним.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, данные, проверки { display-mode: "form" }
import os, re, urllib.request
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

if not os.path.exists("news.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Scr1ptG/ml_for_3p1/main/08-text-exam/data/news.csv", "news.csv")
torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True

print("✅ Готово. Устройство:", device)

---
# Часть 1. Память своими руками

Самая простая память: **новая память = старая память × w + новое слово**. Здесь слово — просто число, а `w` показывает, сколько от старой памяти мы оставляем.

### ✏️ Задание 1. Напишите `remember(xs, w)`: пройдите по числам и верните список памяти после каждого шага

In [ ]:
def remember(xs, w):
    h = 0
    history = []
    for x in xs:
        h = None  # новая память = старая память × w + новое число
        history.append(h)
    return history

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('один шаг', lambda: remember([5], 0.5) == [5], 'В начале память равна 0, поэтому после первого шага h = x'),
    ('затухание', lambda: remember([1, 0, 0, 0], 0.5) == [1, 0.5, 0.25, 0.125], 'Умножайте старую память на w и прибавляйте x'),
    ('накопление', lambda: remember([1, 1, 1], 1) == [1, 2, 3], 'При w = 1 ничего не забывается'),
], 1):
    h = remember([1] + [0] * 19, 0.5)
    plt.bar(range(1, 21), h); plt.title("Сколько осталось от первого слова"); plt.xlabel("шаг"); plt.show()
    print("Через 10 шагов от первого слова осталось", h[9])

<details><summary>💡 Подсказка</summary>

Это одна строка: используйте переменные `h`, `w` и `x`.
</details>

**Проблема забывания:** при `w = 0.5` через 10 слов от первого слова осталось около 0,002. Простая RNN так же быстро забывает начало длинного текста.

А если `w` больше 1, память растёт без остановки. Посмотрите:

### ✏️ Задание 2. Посчитайте `big`: последнюю память для двадцати единиц при `w = 2`

In [ ]:
big = None

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('значение', lambda: big == 2 ** 20 - 1, 'Вызовите remember для списка из 20 единиц и возьмите последний элемент'),
], 2):
    print(f"Память выросла до {big:,}. Это «взрыв»: обучение ломается, loss улетает в огромные числа.")

<details><summary>💡 Подсказка</summary>

Последний элемент списка — индекс `-1`.
</details>

Настоящая RNN устроена так же, только:
* слово — не число, а **вектор** (из `Embedding`);
* `w` — не число, а **матрица весов**, и сеть подбирает её сама;
* результат пропускают через `tanh`, чтобы память не взрывалась.

**Одни и те же веса** применяются на каждом шаге. Поэтому сеть может прочитать текст любой длины.

---
# Часть 2. Слой `nn.RNN`

На вход: батч текстов, в каждом `длина` слов, каждое слово — вектор длины `input_size`. Слой возвращает **две** вещи:
* `out` — память после **каждого** слова;
* `h` — память после **последнего** слова. Её мы и используем для классификации.

In [ ]:
rnn = nn.RNN(input_size=4, hidden_size=3, batch_first=True)
x = torch.randn(2, 5, 4)   # 2 текста, по 5 слов, вектор слова длины 4
out, h = rnn(x)
print("out:", tuple(out.shape))
print("h:  ", tuple(h.shape))

### ✏️ Задание 3. Объясните формы: запишите `out_shape` и `h_shape` как кортежи чисел, не копируя из вывода выше, а подумав

In [ ]:
out_shape = None  # (тексты, слова, память)
h_shape = None  # (слои, тексты, память)

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('out', lambda: out_shape == (2, 5, 3), 'Память (размер hidden_size) после каждого из 5 слов в каждом из 2 текстов'),
    ('h', lambda: h_shape == (1, 2, 3), 'Один слой, 2 текста, память размера 3'),
], 3):
    print("Последняя память из out совпадает с h:", torch.allclose(out[:, -1], h[0]))

<details><summary>💡 Подсказка</summary>

`hidden_size` — размер памяти. В `h` первым идёт число слоёв: у нас он один.
</details>

---
# Часть 3. Данные

Датасет из зачётной: новости трёх категорий. Подготовку вы уже делали, она в скрытой ячейке.

In [ ]:
#@title ⚙️ Запустите: загрузка, токены, словарь, выборки (это вы уже умеете) { display-mode: "form" }
df = pd.read_csv("news.csv").dropna().drop_duplicates("text").reset_index(drop=True)
classes = sorted(df["label"].unique())
y = df["label"].map({c: i for i, c in enumerate(classes)}).values
tokens = [re.findall(r"[а-яёa-z]+", t.lower()) for t in df["text"]]

# Сначала ПЕРЕМЕШИВАЕМ и делим, потом строим словарь
xTrain, xRest, yTrain, yRest = train_test_split(tokens, y, test_size=0.3, random_state=42, stratify=y)
xVal, xTest, yVal, yTest = train_test_split(xRest, yRest, test_size=0.5, random_state=42, stratify=yRest)

counts = Counter(w for t in xTrain for w in t)
word2idx = {w: i + 2 for i, (w, _) in enumerate(counts.most_common(10000))}   # 0 — PAD, 1 — UNK
VOCAB = len(word2idx) + 2
SEQ = 100   # берём первые 100 слов новости

print("Классы:", classes)
print("train / val / test:", len(xTrain), len(xVal), len(xTest))
print("Пример:", " ".join(xTrain[0][:12]), "...")

**Важная деталь для RNN: нули добавляем слева.** Сеть решает по последней памяти. Если нули стоят в конце, последние шаги сеть читает «пустоту» и успевает забыть текст. Если нули в начале, текст идёт последним и свежо помнится.

```
справа: [12, 5, 8, 0, 0, 0]   ← в конце сеть читает пустоту
слева:  [0, 0, 0, 12, 5, 8]   ← в конце настоящие слова
```

### ✏️ Задание 4. Напишите `pad_left(ids)`: оставьте первые SEQ номеров и добавьте нули **слева** до длины SEQ

In [ ]:
def pad_left(ids):
    ids = ids[:SEQ]
    return None

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('короткий', lambda: pad_left([7, 8]) == [0] * (SEQ - 2) + [7, 8], 'Нули стоят перед словами'),
    ('длинный', lambda: pad_left(list(range(1, 300))) == list(range(1, SEQ + 1)), 'Длинный список обрезан до SEQ'),
], 4):
    pass

<details><summary>💡 Подсказка</summary>

Список нулей нужной длины плюс сам список.
</details>

In [ ]:
def to_seq(texts):
    return torch.tensor([pad_left([word2idx.get(w, 1) for w in t]) for t in texts])

sTrain, sVal, sTest = to_seq(xTrain), to_seq(xVal), to_seq(xTest)
print(sTrain.shape)
print(sTrain[0][-10:])

In [ ]:
#@title ⚙️ Запустите: функция обучения (читать не обязательно) { display-mode: "form" }
def accuracy(net, X, Y):
    net.eval()
    with torch.no_grad():
        p = net(X.to(device)).argmax(1).cpu()
    return (p == torch.tensor(Y)).float().mean().item()

def fit(net, epochs, path, lr=0.001):
    """Учит сеть, печатает точность и сохраняет веса лучшей эпохи по val."""
    net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(sTrain, torch.tensor(yTrain)), batch_size=32, shuffle=True)
    hist, best = [], -1
    for ep in range(1, epochs + 1):
        net.train()
        for xb, yb in loader:
            loss = loss_fn(net(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)   # не даём шагу «взорваться»
            opt.step()
        tr, va = accuracy(net, sTrain, yTrain), accuracy(net, sVal, yVal)
        hist.append((tr, va))
        if va > best:
            best = va
            torch.save(net.state_dict(), path)
        if ep % max(1, epochs // 10) == 0 or ep == 1:
            print(f"эпоха {ep:3d}: train {tr:.3f} | val {va:.3f} | лучшая val {best:.3f}")
    return hist, best

def plot_hist(hist, title):
    h = np.array(hist)
    plt.plot(h[:, 0], label="train"); plt.plot(h[:, 1], label="val")
    plt.title(title); plt.xlabel("эпоха"); plt.ylabel("точность"); plt.legend(); plt.show()

print("✅ Готово")

---
# Часть 4. Классификатор на RNN

```
номера слов (батч, 100)
Embedding → (батч, 100, 64)     вектор на каждое слово
RNN       → h: (1, батч, 64)    память после последнего слова
Linear    → (батч, 3)           оценки трёх классов
```

### ✏️ Задание 5. Соберите `RNNNet`: создайте слой RNN и верните ответ по последней памяти

In [ ]:
class RNNNet(nn.Module):
    def __init__(self, emb_dim=64, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, emb_dim, padding_idx=0)
        self.rnn = None  # nn.RNN: вход emb_dim, память hidden, batch_first=True
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(hidden, 3)

    def forward(self, x):
        out, h = self.rnn(self.emb(x))
        return None  # head от последней памяти: h[-1], не забудьте drop

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('слой RNN', lambda: isinstance(RNNNet().rnn, nn.RNN), 'Создайте nn.RNN(...)'),
    ('выход', lambda: RNNNet()(torch.zeros(2, SEQ).long()).shape == (2, 3), 'h[-1] имеет форму (батч, hidden): подайте её в head'),
], 5):
    pass

<details><summary>💡 Подсказка</summary>

Посмотрите, как создавали слой в Части 2: тот же вызов, только другие размеры.
</details>

### ✏️ Задание 6. Обучите RNN не меньше 20 эпох

In [ ]:
rnn_net = RNNNet()
hist_rnn, best_rnn = fit(rnn_net, epochs=None, path="best_rnn.pt")

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('эпохи', lambda: len(hist_rnn) >= 20, 'По заданию нужно минимум 20 эпох'),
    ('учится', lambda: best_rnn > 0.4, 'Наугад было бы 0.33. Проверьте RNNNet'),
], 6):
    plot_hist(hist_rnn, "RNN")

<details><summary>💡 Подсказка</summary>

Вписывается одно число.
</details>

---
# Часть 5. LSTM: память с ластиком

У простой RNN одна «заметка», и каждое слово её переписывает. У **LSTM** есть отдельная **лента памяти** и три вентиля. Вентиль — это число от 0 до 1: 0 значит «закрыто», 1 — «открыто».

| Вентиль | Вопрос | Аналогия |
|---|---|---|
| забывания | что стереть из памяти? | ластик |
| обновления | что записать нового? | карандаш |
| выхода | что из памяти сказать сейчас? | что произнести вслух |

Поэтому LSTM может держать важное слово из начала текста долго. В PyTorch слой возвращает `out, (h, c)`: `h` — то, что сказали вслух, `c` — лента памяти.

### ✏️ Задание 7. Соберите `LSTMNet`: то же самое, но со слоем LSTM

In [ ]:
class LSTMNet(nn.Module):
    def __init__(self, emb_dim=64, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, emb_dim, padding_idx=0)
        self.lstm = None  # nn.LSTM с теми же параметрами, что у RNN
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(hidden, 3)

    def forward(self, x):
        out, h = None, None  # LSTM возвращает out и пару (h, c)
        return self.head(self.drop(h[-1]))

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('слой LSTM', lambda: isinstance(LSTMNet().lstm, nn.LSTM), 'Создайте nn.LSTM(...)'),
    ('выход', lambda: LSTMNet()(torch.zeros(2, SEQ).long()).shape == (2, 3), 'Распакуйте ответ как out, (h, c)'),
], 7):
    pass

<details><summary>💡 Подсказка</summary>

Отличие от RNN в двух местах: имя слоя и то, что вместо `h` приходит пара `(h, c)`.
</details>

Задание LXP: **LSTM 100 эпох без переобучения.** За 100 эпох сеть почти наверняка начнёт переобучаться: точность на train растёт, а на val падает. Нас защищают три вещи:
* `Dropout` в сети и `weight_decay` в оптимизаторе;
* `fit` сохраняет веса **лучшей по val** эпохи, а не последней;
* на графике видно, с какой эпохи началось переобучение.

### ✏️ Задание 8. Обучите LSTM 100 эпох

In [ ]:
lstm_net = LSTMNet()
hist_lstm, best_lstm = fit(lstm_net, epochs=None, path="best_lstm.pt")

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('эпохи', lambda: len(hist_lstm) >= 100, 'По заданию 100 эпох'),
    ('учится', lambda: best_lstm > 0.4, 'Наугад было бы 0.33. Проверьте LSTMNet'),
], 8):
    plot_hist(hist_lstm, "LSTM")

<details><summary>💡 Подсказка</summary>

Обучение займёт несколько минут. С GPU быстрее.
</details>

**Дообучение:** если вызвать `fit` для той же сети ещё раз, она продолжит учиться с текущих весов, а не с нуля. Так можно добавить эпох, не начиная заново. Перед этим удобно загрузить лучшие веса.

---
# Часть 6. Честная проверка на test

### ✏️ Задание 9. Загрузите лучшие веса обеих сетей и посчитайте `acc_rnn` и `acc_lstm` на test

In [ ]:
rnn_net.load_state_dict(torch.load("best_rnn.pt", map_location=device))
pass  # то же для lstm_net

acc_rnn = None  # функция accuracy(сеть, X, Y)
acc_lstm = None

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('RNN', lambda: 0.4 < acc_rnn <= 1, 'accuracy(rnn_net, sTest, yTest)'),
    ('LSTM', lambda: 0.4 < acc_lstm <= 1, 'Сначала загрузите веса best_lstm.pt'),
], 9):
    print(f"RNN:  {acc_rnn:.3f}")
    print(f"LSTM: {acc_lstm:.3f}")

<details><summary>💡 Подсказка</summary>

Загрузка весов LSTM — такая же строка, как для RNN выше.
</details>

**Вывод (2–3 предложения):** какая сеть лучше? На какой эпохе LSTM начала переобучаться? Почему для RNN нули ставили слева?

_Ваш ответ:_

---
## 🏁 Итог
* RNN читает текст по слову и хранит **память**, одни и те же веса на каждом шаге.
* Простая память быстро **забывает** начало или **взрывается**.
* LSTM добавляет **ленту памяти** и вентили: забыть, записать, сказать.
* Сравниваем по val, сохраняем лучшую эпоху, test проверяем один раз.